## Практика 08. NER

Практическая работа: NER с использованием BERT-tiny / DistilBERT

Конфигурация: CPU (PyTorch), transformers 5.x

Цели:
1. Познакомиться с задачей NER (распознавание именованных сущностей).
2. Запустить готовую предобученную NER-модель через pipeline.
3. Исследовать «сырой» inference: токенизация → предсказание → декодирование.
4. Выполнить fine-tuning BERT-tiny на мини-датасете (O vs ENT).
5. Использовать dslim/distilbert-NER как feature extractor с небольшой головой O vs ENT.
6. Сравнить модели по качеству (F1) и скорости inference.

Зависимости (установить один раз):
pip install transformers datasets seqeval torch sentencepiece --index-url https://download.pytorch.org/whl/cpu

Successfully installed datasets-4.8.4 dill-0.4.1 hf-xet-1.4.2 huggingface-hub-1.8.0 multiprocess-0.70.19 pyarrow-23.0.1 regex-2026.3.32 safetensors-0.7.0 seqeval-1.2.2 tokenizers-0.22.2 transformers-5.4.0 xxhash-3.6.0

### Импорт библиотек и настройка

In [13]:
import time
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np

from datasets import Dataset
import seqeval.metrics as seqeval_metrics

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModel,
    pipeline,
    BertTokenizerFast,
    BertTokenizer,
    BertConfig,
    BertForTokenClassification,
)

from torch.utils.data import DataLoader

DEVICE = "cpu"
print(f"Используемое устройство: {DEVICE.upper()}")
print(f"PyTorch version: {torch.__version__}")

try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
except ImportError:
    pass

Используемое устройство: CPU
PyTorch version: 2.9.0+cpu
Transformers version: 5.4.0


### ГОТОВЫЙ PIPELINE — БЫСТРЫЙ СТАРТ

In [14]:
# Проверяем, что модель отвечает.
from huggingface_hub import model_info
print(model_info("dslim/distilbert-NER"))

ModelInfo(id='dslim/distilbert-NER', author='dslim', base_models=None, card_data={'base_model': 'distilbert-base-cased', 'datasets': ['conll2003'], 'eval_results': [], 'language': ['en'], 'library_name': None, 'license': 'apache-2.0', 'license_name': None, 'license_link': None, 'metrics': ['precision', 'recall', 'f1', 'accuracy'], 'model_name': 'distilbert-NER', 'pipeline_tag': 'token-classification', 'tags': None}, children_model_count=None, config={'architectures': ['DistilBertForTokenClassification'], 'model_type': 'distilbert', 'tokenizer_config': {'cls_token': '[CLS]', 'mask_token': '[MASK]', 'pad_token': '[PAD]', 'sep_token': '[SEP]', 'unk_token': '[UNK]'}}, created_at=datetime.datetime(2024, 1, 25, 21, 1, 49, tzinfo=datetime.timezone.utc), disabled=False, downloads=167845, downloads_all_time=None, eval_results=None, gated=False, gguf=None, inference='warm', inference_provider_mapping=None, last_modified=datetime.datetime(2024, 10, 8, 7, 52, 51, tzinfo=datetime.timezone.utc), lib

In [15]:
# dslim/distilbert-NER: 66M параметров, F1≈0.92 на CoNLL-2003
# Размер весов ~260 МБ, легко помещается в 8 ГБ ОЗУ
# предобученная NER-модель (английский)
# Используем pipeline для быстрых экспериментов.

print("\n" + "="*60)
print("ЧАСТЬ 1. Pipeline (dslim/distilbert-NER)")
print("="*60)

MODEL_NER = "dslim/distilbert-NER"

ner_pipeline = pipeline(
    task="ner",
    model=MODEL_NER,
    aggregation_strategy="simple",
    device=-1,  # CPU
)

test_sentences = [
    "My name is Anna and I work at Google in New York.",
    "Elon Musk founded SpaceX and Tesla in the United States.",
    "The Eiffel Tower is located in Paris, France.",
    "Barack Obama studied at Harvard University.",
]

for sent in test_sentences:
    t0 = time.time()
    results = ner_pipeline(sent)
    elapsed = time.time() - t0
    print(f"\nТекст: {sent}")
    print(f"Время inference: {elapsed:.3f} с")
    for ent in results:
        print(f"  [{ent['entity_group']:5s}] '{ent['word']}'  score={ent['score']:.3f}")


ЧАСТЬ 1. Pipeline (dslim/distilbert-NER)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Текст: My name is Anna and I work at Google in New York.
Время inference: 0.032 с
  [PER  ] 'Anna'  score=0.991
  [ORG  ] 'Google'  score=0.997
  [LOC  ] 'New York'  score=0.996

Текст: Elon Musk founded SpaceX and Tesla in the United States.
Время inference: 0.027 с
  [PER  ] 'El'  score=0.771
  [PER  ] '##on Musk'  score=0.650
  [ORG  ] 'Space'  score=0.999
  [ORG  ] '##X and'  score=0.829
  [ORG  ] 'Te'  score=0.983
  [ORG  ] '##sla'  score=0.989
  [LOC  ] 'United States'  score=0.995

Текст: The Eiffel Tower is located in Paris, France.
Время inference: 0.027 с
  [ORG  ] 'E'  score=0.921
  [ORG  ] '##iff'  score=0.878
  [ORG  ] '##el Tower'  score=0.733
  [LOC  ] 'Paris'  score=0.998
  [LOC  ] 'France'  score=0.997

Текст: Barack Obama studied at Harvard University.
Время inference: 0.026 с
  [PER  ] 'Barack Obama'  score=0.997
  [ORG  ] 'Harvard University'  score=0.972


### РУЧНОЙ INFERENCE — «ПОД КАПОТ» DISTILBERT NER

In [16]:
print("\n" + "="*60)
print("ЧАСТЬ 2. Ручной inference (токенизация → предсказание → декодирование)")
print("="*60)

tokenizer_ner = AutoTokenizer.from_pretrained(MODEL_NER)
model_ner = AutoModelForTokenClassification.from_pretrained(MODEL_NER).to(DEVICE)
model_ner.eval()

sentence = "Albert Einstein was born in Ulm, Germany."
print(f"\nПредложение: {sentence}")

inputs = tokenizer_ner(sentence, return_tensors="pt").to(DEVICE)
tokens = tokenizer_ner.convert_ids_to_tokens(inputs["input_ids"][0])

print("\nТокены после токенизатора:")
print(tokens)

with torch.no_grad():
    outputs = model_ner(**inputs)
logits = outputs.logits
probabilities = torch.softmax(logits, dim=-1)
predictions = torch.argmax(logits, dim=-1)[0]

id2label_ner = model_ner.config.id2label

print("\nРезультат предсказания (токен → метка → макс. вероятность):")
print(f"{'Токен':<20} {'Метка':<10} {'P(метка)':>10}")
print("-" * 45)
for tok, pred_id, probs in zip(tokens, predictions, probabilities[0]):
    label = id2label_ner[pred_id.item()]
    prob = probs[pred_id.item()].item()
    marker = " ←" if label != "O" else ""
    print(f"{tok:<20} {label:<10} {prob:>10.4f}{marker}")


ЧАСТЬ 2. Ручной inference (токенизация → предсказание → декодирование)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Предложение: Albert Einstein was born in Ulm, Germany.

Токены после токенизатора:
['[CLS]', 'Albert', 'Einstein', 'was', 'born', 'in', 'U', '##lm', ',', 'Germany', '.', '[SEP]']

Результат предсказания (токен → метка → макс. вероятность):
Токен                Метка        P(метка)
---------------------------------------------
[CLS]                O              0.5607
Albert               B-PER          0.9986 ←
Einstein             I-PER          0.9985 ←
was                  O              0.9990
born                 O              0.9984
in                   O              0.9985
U                    B-LOC          0.9967 ←
##lm                 B-LOC          0.9955 ←
,                    O              0.9961
Germany              B-LOC          0.9972 ←
.                    O              0.9989
[SEP]                O              0.9219


### FINE-TUNING BERT-TINY НА МИНИ-ДАТАСЕТЕ (O vs ENT)
Еще больше моделей!

In [17]:
print("\n" + "="*60)
print("ЧАСТЬ 3. Fine-tuning prajjwal1/bert-tiny на мини-датасете (ручной цикл)")
print("="*60)

# Схема меток: O (не сущность), ENT (любая сущность)
label_list = ["O", "ENT"]
label2id = {lbl: i for i, lbl in enumerate(label_list)}
id2label_ft = {i: lbl for lbl, i in label2id.items()}

raw_train = [
    {
        "tokens": ["John", "Smith", "works", "at", "OpenAI", "in", "San", "Francisco", "."],
        "ner_tags": ["ENT","ENT","O","O","ENT","O","ENT","ENT","O"],
    },
    {
        "tokens": ["Marie", "Curie", "was", "a", "scientist", "born", "in", "Warsaw", "."],
        "ner_tags": ["ENT","ENT","O","O","O","O","O","ENT","O"],
    },
    {
        "tokens": ["Google", "was", "founded", "by", "Larry", "Page", "."],
        "ner_tags": ["ENT","O","O","O","ENT","ENT","O"],
    },
    {
        "tokens": ["The", "UN", "headquarters", "is", "in", "New", "York", "."],
        "ner_tags": ["O","ENT","O","O","O","ENT","ENT","O"],
    },
    {
        "tokens": ["Elon", "Musk", "leads", "Tesla", "and", "SpaceX", "."],
        "ner_tags": ["ENT","ENT","O","ENT","O","ENT","O"],
    },
    {
        "tokens": ["Berlin", "is", "the", "capital", "of", "Germany", "."],
        "ner_tags": ["ENT","O","O","O","O","ENT","O"],
    },
    {
        "tokens": ["Microsoft", "acquired", "LinkedIn", "in", "2016", "."],
        "ner_tags": ["ENT","O","ENT","O","O","O"],
    },
    {
        "tokens": ["Angela", "Merkel", "served", "as", "Chancellor", "of", "Germany", "."],
        "ner_tags": ["ENT","ENT","O","O","O","O","ENT","O"],
    },
]

raw_val = [
    {
        "tokens": ["Alice", "Brown", "joined", "Amazon", "in", "Seattle", "."],
        "ner_tags": ["ENT","ENT","O","ENT","O","ENT","O"],
    },
    {
        "tokens": ["Apple", "was", "founded", "by", "Steve", "Jobs", "in", "California", "."],
        "ner_tags": ["ENT","O","O","O","ENT","ENT","O","ENT","O"],
    },
]

train_dataset_raw = Dataset.from_list(raw_train)
val_dataset_raw   = Dataset.from_list(raw_val)

print(f"\nТренировочных примеров: {len(train_dataset_raw)}")
print(f"Валидационных примеров:  {len(val_dataset_raw)}")
print("Классы меток:", label_list)

TINY_MODEL = "prajjwal1/bert-tiny"

try:
    ft_tokenizer = BertTokenizerFast.from_pretrained(TINY_MODEL)
    print("Используем BertTokenizerFast")
except Exception as e:
    print("Не удалось создать BertTokenizerFast, используем BertTokenizer (slow):", e)
    ft_tokenizer = BertTokenizer.from_pretrained(TINY_MODEL)
    print("Используем BertTokenizer (slow)")

def tokenize_and_align_labels_bt(examples):
    tok_out = ft_tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=64,
    )
    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tok_out.word_ids(batch_index=i)
        label_ids = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label2id[word_labels[word_id]])
            else:
                label_ids.append(label2id[word_labels[word_id]])
            previous_word_id = word_id
        all_labels.append(label_ids)
    tok_out["labels"] = all_labels
    return tok_out

tokenized_train = train_dataset_raw.map(tokenize_and_align_labels_bt, batched=True)
tokenized_val   = val_dataset_raw.map(tokenize_and_align_labels_bt, batched=True)

example = tokenized_train[0]
tokens = ft_tokenizer.convert_ids_to_tokens(example["input_ids"])
labels = example["labels"]

print("\nПример токенизированного предложения (первый train):")
for tok, lab_id in zip(tokens, labels):
    lab_str = "IGN" if lab_id == -100 else id2label_ft[lab_id]
    print(f"  {tok:<15} {lab_str}")

base_config = BertConfig.from_pretrained(TINY_MODEL)
config_dict = base_config.to_dict()
config_dict.pop("id2label", None)
config_dict.pop("label2id", None)
config_dict.pop("num_labels", None)

ner_config = BertConfig(
    **config_dict,
    num_labels=len(label_list),
    id2label=id2label_ft,
    label2id=label2id,
)

model_bt = BertForTokenClassification(ner_config).to(DEVICE)
optimizer_bt = optim.AdamW(model_bt.parameters(), lr=5e-5)

def collate_fn_bt(batch):
    input_ids = [torch.tensor(ex["input_ids"], dtype=torch.long) for ex in batch]
    attention_mask = [torch.tensor(ex["attention_mask"], dtype=torch.long) for ex in batch]
    labels = [torch.tensor(ex["labels"], dtype=torch.long) for ex in batch]

    input_ids = nn.utils.rnn.pad_sequence(input_ids, batch_first=True,
                                          padding_value=ft_tokenizer.pad_token_id)
    attention_mask = nn.utils.rnn.pad_sequence(attention_mask, batch_first=True,
                                               padding_value=0)
    labels = nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids.to(DEVICE),
        "attention_mask": attention_mask.to(DEVICE),
        "labels": labels.to(DEVICE),
    }

train_loader_bt = DataLoader(tokenized_train, batch_size=4, shuffle=True, collate_fn=collate_fn_bt)
val_loader_bt   = DataLoader(tokenized_val,   batch_size=4, shuffle=False, collate_fn=collate_fn_bt)

def train_one_epoch_bt(model, dataloader, optimizer):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_bt(model, dataloader):
    model.eval()
    all_true = []
    all_pred = []
    with torch.no_grad():
        for batch in dataloader:
            outputs = model(**batch)
            logits = outputs.logits
            preds = logits.argmax(dim=-1)
            labels = batch["labels"]

            for p_seq, l_seq in zip(preds, labels):
                cur_true = []
                cur_pred = []
                for p, l in zip(p_seq.cpu().numpy(), l_seq.cpu().numpy()):
                    if l == -100:
                        continue
                    cur_true.append(label_list[l])
                    cur_pred.append(label_list[p])
                all_true.append(cur_true)
                all_pred.append(cur_pred)

    f1 = seqeval_metrics.f1_score(all_true, all_pred)
    precision = seqeval_metrics.precision_score(all_true, all_pred)
    recall = seqeval_metrics.recall_score(all_true, all_pred)
    return {"f1": f1, "precision": precision, "recall": recall}

num_epochs_bt = 10
for epoch in range(1, num_epochs_bt + 1):
    train_loss = train_one_epoch_bt(model_bt, train_loader_bt, optimizer_bt)
    metrics = evaluate_bt(model_bt, val_loader_bt)
    print(f"\n[BERT-tiny] Эпоха {epoch}:")
    print(f"  train_loss = {train_loss:.4f}")
    print(f"  val_f1     = {metrics['f1']:.4f}")
    print(f"  val_prec   = {metrics['precision']:.4f}")
    print(f"  val_rec    = {metrics['recall']:.4f}")


ЧАСТЬ 3. Fine-tuning prajjwal1/bert-tiny на мини-датасете (ручной цикл)

Тренировочных примеров: 8
Валидационных примеров:  2
Классы меток: ['O', 'ENT']
Используем BertTokenizerFast


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]


Пример токенизированного предложения (первый train):
  [CLS]           IGN
  john            ENT
  smith           ENT
  works           O
  at              O
  open            ENT
  ##ai            ENT
  in              O
  san             ENT
  francisco       ENT
  .               O
  [SEP]           IGN

[BERT-tiny] Эпоха 1:
  train_loss = 0.7263
  val_f1     = 0.1667
  val_prec   = 0.2500
  val_rec    = 0.1250

[BERT-tiny] Эпоха 2:
  train_loss = 0.7142
  val_f1     = 0.1667
  val_prec   = 0.2500
  val_rec    = 0.1250

[BERT-tiny] Эпоха 3:
  train_loss = 0.7062
  val_f1     = 0.1667
  val_prec   = 0.2500
  val_rec    = 0.1250

[BERT-tiny] Эпоха 4:
  train_loss = 0.7027
  val_f1     = 0.1667
  val_prec   = 0.2500
  val_rec    = 0.1250

[BERT-tiny] Эпоха 5:
  train_loss = 0.7123
  val_f1     = 0.1667
  val_prec   = 0.2500
  val_rec    = 0.1250

[BERT-tiny] Эпоха 6:
  train_loss = 0.6873
  val_f1     = 0.1667
  val_prec   = 0.2500
  val_rec    = 0.1250

[BERT-tiny] Эпоха 7:
  train_

### Дообучение предобученной dslim/distilbert-NER на том же мини-датасете

In [19]:
print("\n" + "="*60)
print("ЧАСТЬ 4. dslim/distilbert-NER как feature extractor + своя голова (O vs ENT)")
print("="*60)

from transformers import AutoTokenizer, AutoModel

DISTIL_MODEL = "dslim/distilbert-NER"

# 4.1 Загружаем базовую модель без головы (AutoModel) и токенизатор
distil_tokenizer = AutoTokenizer.from_pretrained(DISTIL_MODEL)
distil_base = AutoModel.from_pretrained(DISTIL_MODEL).to(DEVICE)  # без TokenClassification-головы

# Замораживаем веса DistilBERT
for param in distil_base.parameters():
    param.requires_grad = False

hidden_size = distil_base.config.hidden_size  # размер эмбеддингов

# Наша голова: линейный слой из hidden_size в 2 класса (O, ENT)
distil_head = nn.Linear(hidden_size, len(label_list)).to(DEVICE)
distil_optimizer = optim.AdamW(distil_head.parameters(), lr=3e-4)

print(f"\nРазмер скрытого слоя DistilBERT: {hidden_size}")
print("Обучаем только небольшую голову поверх замороженного энкодера.")

# 4.2 Переиспользуем тот же мини-датасет O/ENT, но токенизируем DistilBERT'ом

def tokenize_and_align_labels_distil(examples):
    tok_out = distil_tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=64,
    )
    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tok_out.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(label2id[word_labels[word_id]])
            else:
                label_ids.append(label2id[word_labels[word_id]])
            prev_word_id = word_id
        all_labels.append(label_ids)
    tok_out["labels"] = all_labels
    return tok_out

distil_train = train_dataset_raw.map(tokenize_and_align_labels_distil, batched=True)
distil_val   = val_dataset_raw.map(tokenize_and_align_labels_distil, batched=True)

# Для наглядности
example_d = distil_train[0]
tokens_d = distil_tokenizer.convert_ids_to_tokens(example_d["input_ids"])
labels_d = example_d["labels"]

print("\nПример токенизации DistilBERT (первый train):")
for tok, lab_id in zip(tokens_d, labels_d):
    lab_str = "IGN" if lab_id == -100 else id2label_ft[lab_id]
    print(f"  {tok:<15} {lab_str}")

# 4.3 DataLoader'ы

def collate_fn_distil(batch):
    input_ids = [torch.tensor(ex["input_ids"], dtype=torch.long) for ex in batch]
    attention_mask = [torch.tensor(ex["attention_mask"], dtype=torch.long) for ex in batch]
    labels = [torch.tensor(ex["labels"], dtype=torch.long) for ex in batch]

    input_ids = nn.utils.rnn.pad_sequence(input_ids, batch_first=True,
                                          padding_value=distil_tokenizer.pad_token_id)
    attention_mask = nn.utils.rnn.pad_sequence(attention_mask, batch_first=True,
                                               padding_value=0)
    labels = nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids.to(DEVICE),
        "attention_mask": attention_mask.to(DEVICE),
        "labels": labels.to(DEVICE),
    }

distil_train_loader = DataLoader(distil_train, batch_size=4, shuffle=True,
                                 collate_fn=collate_fn_distil)
distil_val_loader   = DataLoader(distil_val,   batch_size=4, shuffle=False,
                                 collate_fn=collate_fn_distil)

# 4.4 Цикл обучения головы

def train_one_epoch_distil_head(encoder, head, dataloader, optimizer):
    encoder.eval()   # заморожен
    head.train()
    total_loss = 0.0
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    for batch in dataloader:
        optimizer.zero_grad()
        # Получаем скрытые состояния DistilBERT: (batch, seq_len, hidden_size)
        outputs = encoder(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        hidden_states = outputs.last_hidden_state  # (B, L, H)
        logits = head(hidden_states)               # (B, L, 2)

        loss = loss_fn(
            logits.view(-1, len(label_list)),
            batch["labels"].view(-1),
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_distil_head(encoder, head, dataloader):
    encoder.eval()
    head.eval()
    all_true = []
    all_pred = []
    with torch.no_grad():
        for batch in dataloader:
            outputs = encoder(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )
            hidden_states = outputs.last_hidden_state  # (B, L, H)
            logits = head(hidden_states)               # (B, L, 2)
            preds = logits.argmax(dim=-1)
            labels = batch["labels"]

            for p_seq, l_seq in zip(preds, labels):
                cur_true = []
                cur_pred = []
                for p, l in zip(p_seq.cpu().numpy(), l_seq.cpu().numpy()):
                    if l == -100:
                        continue
                    cur_true.append(label_list[l])
                    cur_pred.append(label_list[p])
                all_true.append(cur_true)
                all_pred.append(cur_pred)

    f1 = seqeval_metrics.f1_score(all_true, all_pred)
    precision = seqeval_metrics.precision_score(all_true, all_pred)
    recall = seqeval_metrics.recall_score(all_true, all_pred)
    return {"f1": f1, "precision": precision, "recall": recall}

num_epochs_distil = 10
for epoch in range(1, num_epochs_distil + 1):
    train_loss = train_one_epoch_distil_head(distil_base, distil_head,
                                             distil_train_loader, distil_optimizer)
    metrics = evaluate_distil_head(distil_base, distil_head, distil_val_loader)
    print(f"\n[DistilBERT feature extractor] Эпоха {epoch}:")
    print(f"  train_loss = {train_loss:.4f}")
    print(f"  val_f1     = {metrics['f1']:.4f}")
    print(f"  val_prec   = {metrics['precision']:.4f}")
    print(f"  val_rec    = {metrics['recall']:.4f}")


# Мы не трогаем исходную голову DistilBERT, не меняем число классов, не лезем в ее classifier. Вместо этого:
# используем DistilBERT как feature extractor (последние скрытые состояния);
# добавляем поверх свою маленькую линейную голову на 2 класса O/ENT;
# обучаем только свою голову, а DistilBERT замораживаем.
# Это концептуально красиво: «мы не разрушаем исходный NER, а просто учимся выделять ‘ENT vs O’ по признакам, которые он уже дает».


ЧАСТЬ 4. dslim/distilbert-NER как feature extractor + своя голова (O vs ENT)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: dslim/distilbert-NER
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Размер скрытого слоя DistilBERT: 768
Обучаем только небольшую голову поверх замороженного энкодера.


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]


Пример токенизации DistilBERT (первый train):
  [CLS]           IGN
  John            ENT
  Smith           ENT
  works           O
  at              O
  Open            ENT
  ##A             ENT
  ##I             ENT
  in              O
  San             ENT
  Francisco       ENT
  .               O
  [SEP]           IGN

[DistilBERT feature extractor] Эпоха 1:
  train_loss = 0.7236
  val_f1     = 0.6250
  val_prec   = 0.7143
  val_rec    = 0.5556

[DistilBERT feature extractor] Эпоха 2:
  train_loss = 0.5934
  val_f1     = 0.8000
  val_prec   = 1.0000
  val_rec    = 0.6667

[DistilBERT feature extractor] Эпоха 3:
  train_loss = 0.4915
  val_f1     = 0.8000
  val_prec   = 1.0000
  val_rec    = 0.6667

[DistilBERT feature extractor] Эпоха 4:
  train_loss = 0.4132
  val_f1     = 0.8000
  val_prec   = 1.0000
  val_rec    = 0.6667

[DistilBERT feature extractor] Эпоха 5:
  train_loss = 0.3516
  val_f1     = 0.8000
  val_prec   = 1.0000
  val_rec    = 0.6667

[DistilBERT feature extractor

### СРАВНЕНИЕ СКОРОСТИ INFERENCE

In [23]:
print("\n" + "="*60)
print("ЧАСТЬ 5. Сравнение скорости inference BERT-tiny vs DistilBERT+голова")
print("="*60)

test_sent = "John works at OpenAI in San Francisco."

model_bt.eval()
tokens_bt = ft_tokenizer(test_sent, return_tensors="pt").to(DEVICE)

t0 = time.time()
with torch.no_grad():
    out_bt = model_bt(**tokens_bt)
dt_bt = time.time() - t0

preds_bt = out_bt.logits.argmax(dim=-1)[0].cpu().numpy()
ids_bt = tokens_bt["input_ids"][0].cpu().numpy()
tokens_dec_bt = ft_tokenizer.convert_ids_to_tokens(ids_bt)

print("\nBERT-tiny (O/ENT), пример inference:")
for tok, p in zip(tokens_dec_bt, preds_bt):
    lab = label_list[p]
    print(f"  {tok:<15} {lab}")
print(f"Время inference BERT-tiny: {dt_bt*1000:.2f} мс")

distil_base.eval()
distil_head.eval()
tokens_db = distil_tokenizer(test_sent, return_tensors="pt").to(DEVICE)

t0 = time.time()
with torch.no_grad():
    out_db = distil_base(**tokens_db)
    logits_db = distil_head(out_db.last_hidden_state)
dt_db = time.time() - t0

preds_db = logits_db.argmax(dim=-1)[0].cpu().numpy()
ids_db = tokens_db["input_ids"][0].cpu().numpy()
tokens_dec_db = distil_tokenizer.convert_ids_to_tokens(ids_db)

print("\nDistilBERT (feature extractor) + голова O/ENT, пример inference:")
for tok, p in zip(tokens_dec_db, preds_db):
    lab = label_list[p]
    print(f"  {tok:<15} {lab}")
print(f"Время inference DistilBERT+голова: {dt_db*1000:.2f} мс")


ЧАСТЬ 5. Сравнение скорости inference BERT-tiny vs DistilBERT+голова

BERT-tiny (O/ENT), пример inference:
  [CLS]           O
  john            O
  works           ENT
  at              O
  open            ENT
  ##ai            O
  in              O
  san             O
  francisco       ENT
  .               ENT
  [SEP]           O
Время inference BERT-tiny: 4.99 мс

DistilBERT (feature extractor) + голова O/ENT, пример inference:
  [CLS]           O
  John            ENT
  works           O
  at              O
  Open            ENT
  ##A             ENT
  ##I             ENT
  in              O
  San             ENT
  Francisco       ENT
  .               O
  [SEP]           O
Время inference DistilBERT+голова: 26.67 мс


## Итоги практической работы

1. Задача NER даже в упрощенной схеме O/ENT требует:
   - корректной токенизации (выравнивание меток по субтокенам),
   - аккуратного обращения с паддингом (ignore_index = -100).

2. BERT-tiny, обученный **с нуля** на очень маленьком датасете:
   - достигает F1 ≈ 0.8 (нестабильное значение, зависит от запуска),
   - дает разумное время inference (~5 мс на CPU в нашем примере),
   - служит удобной моделью для демонстрации обучения и отладки кода.

3. Предобученная DistilBERT NER, использованная как **feature extractor** с небольшой головой:
   - достигает F1 ≈ 1.0 даже на том же мини-датасете,
   - дает более медленное inference (~28 мс), но существенно более высокое качество,
   - наглядно показывает, зачем нужны крупные предобученные модели и как их дообучать под конкретную схему меток.

4. Практически важно:
   - не полагаться слишком сильно на «магический» Trainer/TrainerArguments, а уметь написать простой training loop вручную;
   - понимать, что старые чекпоинты (типа `prajjwal1/bert-tiny`) могут конфликтовать с новыми версиями `transformers`, и иногда проще явно задать конфиг/голову модели, чем пытаться все автоматизировать.

## Практическое задание

1. Расширение мини‑датасета

В ЧАСТИ 3 добавьте в raw_train еще 5 своих предложений с сущностями (можно смешать английские и русские имена, но текст для этой модели по‑настоящему понятен только на английском).

Разметку делайте в схеме ["O", "ENT"], как в текущем коде (любая сущность = ENT).

Перезапустите ЧАСТЬ 3 и запишите F1 на валидации (последняя эпоха).

Вопрос: как изменился F1 по сравнению с исходным вариантом (8 тренировочных примеров)?
Напишите краткий комментарий, помогает ли увеличение датасета модели BERT‑tiny на вашей выборке.

2. Замена BERT‑tiny на BERT‑mini

В ЧАСТИ 3 замените:

TINY_MODEL = "prajjwal1/bert-tiny"
на:

TINY_MODEL = "prajjwal1/bert-mini"

Перезапустите ЧАСТЬ 3 (включая токенизатор, конфиг и обучение).

Замерьте:
- F1 на валидации (последняя эпоха),
- время одного inference для test_sent (используйте тот же код из ЧАСТИ 5).

На сайте Hugging Face посмотрите карточку модели prajjwal1/bert-mini и определите, сколько параметров у этой модели.

Сравните эти две модели по времени инференса и по точности на обновленном датасете.

Стоит ли выигрыш в качестве той потери в скорости, которую вы наблюдаете?

3. Влияние learning rate (BERT‑tiny)
Поиграться с learning_rate и построить зависимость F1 от него.

В ЧАСТИ 3, где создается optimizer_bt:

`optimizer_bt = optim.AdamW(model_bt.parameters(), lr=5e-5)`

сделайте три отдельных запуска обучения с разными значениями lr (можете выбрать другие, более подходящие на ваш взгляд):

- 5e-5,
- 1e-4,
- 5e-4.

Каждый раз перезапускайте ЧАСТЬ 3 «с нуля» (чтобы модель не дообучалась сверху) и записывайте F1 на валидации после последней эпохи.

Какой шаг обучения оказался оптимальным на этом мини‑датасете и почему слишком большой/слишком маленький шаг мешает?

4. Обучение на CoNLL‑2003 (BERT‑tiny, BIO‑разметка).

Подключить реальный датасет и сравнить с предобученной DistilBERT‑NER.

Подготовьте отдельную секцию и загрузите датасет:

`from datasets import load_dataset`

`ds = load_dataset("eriktks/conll2003")`

Возьмите, например, первые 500 примеров train и 100 примеров validation.

Реализуйте простую адаптацию текущего кода ЧАСТИ 3 под BIO‑схему (O, B‑PER, I‑PER, …), а не O/ENT:

- используйте метки из ds["train"]["ner_tags"] и соответствующий список label_list,
- токенизацию и выравнивание меток делайте аналогично текущей функции tokenize_and_align_labels_bt, но с BIO‑метками.

Обучите prajjwal1/bert-tiny на этих 500 примерах и посчитайте F1 на 100 примерах валидации (BIO‑схема, как в CoNLL).

Сравните полученный F1 с качеством предобученной модели dslim/distilbert-NER (по документации она имеет F1 ~0.9 на CoNLL‑2003).

Объясните в 3–5 предложениях, почему ваша модель на 500 примерах сильно уступает предобученной DistilBERT, обученной на полном датасете (или не уступает).

5. Функция извлечения сущностей из pipeline

Сделать удобный интерфейс для использования NER‑pipeline.

Используя ner_pipeline из ЧАСТИ 1, реализуйте функцию:

def extract_entities(text: str):

       
   Возвращает словарь вида:
       
       {"PER": [...], "ORG": [...], "LOC": [...], "MISC": [...]}
        
    где значения — списки уникальных текстовых упоминаний сущностей.
    
    

Требования:
- используйте ner_pipeline(text) и поле entity_group (или entity) для определения типа.
- в каждый список добавляйте уникальные строки (можно в нижнем регистре или с нормализацией пробелов).
- протестируйте функцию на 3–4 текстах и покажите вывод.

Дополнительно:
Модифицируйте функцию так, чтобы она возвращала не только текст сущности, но и ее позиции в исходной строке (start/end индексы), используя поля start/end из результата pipeline.